# 6.2 衰减校正算法

**学习目标**
- 理解PHI方法进行衰减校正的原理
- 观察校正前后反射率的变化
- 理解KDP辅助校正的优势

**参考文献**：Ryzhkov & Zrnic, Chapter 6, pp. 596-630

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def specific_attenuation(R, wavelength_mm):
    """比衰减 (dB/km)"""
    if wavelength_mm < 4:  # X
        return 0.00059 * R**0.79
    elif wavelength_mm < 8:  # C
        return 0.000187 * R**0.82
    else:  # S
        return 0.000138 * R**0.85

def phi_dp_to_kdp(phidp, dr_km):
    """从PHI计算KDP"""
    return np.gradient(phidp, dr_km)

def attenuation_correction():
    """演示衰减校正"""
    # 模拟雷达观测
    range_km = np.linspace(5, 150, 100)
    dr = range_km[1] - range_km[0]
    
    # 假设真实Z恒定（均匀降水）
    Z_true = 40  # dBZ
    R = 20  # mm/h
    wavelength_mm = 32  # X-band
    
    # 累积衰减
    gamma = specific_attenuation(R, wavelength_mm)
    cumulative_atten = np.zeros_like(range_km)
    for i in range(1, len(range_km)):
        cumulative_atten[i] = cumulative_atten[i-1] + gamma * dr
    
    # 测量Z（衰减后）
    Z_measured = Z_true - cumulative_atten
    
    # 校正（简单方法）
    Z_corrected_simple = Z_measured + cumulative_atten
    
    # PHI方法校正
    # 假设PHI累积与衰减成正比
    PHI_dp = cumulative_atten * 2  # 简化关系
    Z_corrected_phi = Z_measured + 0.5 * PHI_dp
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(range_km, Z_true * np.ones_like(range_km), 'k--', linewidth=2, label='真实 Z')
    ax.plot(range_km, Z_measured, 'r-', linewidth=2, label='测量 Z (衰减后)')
    ax.plot(range_km, Z_corrected_simple, 'b:', linewidth=2, label='简单校正')
    ax.plot(range_km, Z_corrected_phi, 'g-', linewidth=2, label='PHI方法校正')
    ax.fill_between(range_km, Z_measured, Z_true, alpha=0.2, color='red')
    ax.set_xlabel('距离 (km)', fontsize=12)
    ax.set_ylabel('反射率 Z (dBZ)', fontsize=12)
    ax.set_title(f'X波段衰减校正示意 (R={R}mm/h, γ={gamma:.4f}dB/km)', fontsize=13)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    ax.set_xlim(5, 150)
    ax.set_ylim(20, 45)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== 衰减校正统计 ===")
    print(f"比衰减 γ = {gamma:.4f} dB/km")
    print(f"在100km处总衰减 = {cumulative_atten[-1]:.1f} dB")
    print(f"校正后误差（简单法）= ±{np.std(Z_corrected_simple - Z_true):.1f} dB")
    print(f"校正后误差（PHI法）= ±{np.std(Z_corrected_phi - Z_true):.1f} dB")

In [ ]:
attenuation_correction()

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 6, Springer.